In [1]:
import numpy as np
import pandas as pd

In [2]:
def import_excel(start_year, end_year):
    df = pd.read_excel('./rent/rent_combined_2008-2020.xlsx', sheet_name=[str(i) for i in range(start_year, end_year+1)])  
    return df

In [3]:
rent = import_excel(2008, 2020)

In [4]:
plz = pd.read_excel('./plz/plz_per_lor.xls', sheet_name='all')

plz['LOR'] = plz['LOR'].apply('{:0>6}'.format)
plz['PR'] = plz['PR'].apply('{:0>8}'.format)

plz.drop(['Planungsraum', 'Planungsraum Name'], axis=1, inplace=True)

In [5]:
col_mapping = {
    'Wohnungsgröße in m2': 'apt_size',
    'Kaltmiete alle Marktsegmente in €/m2/Monat': 'sqm_price_all',
    'Wohnkosten kalt in €/Monat': 'rent_cold',
    'Wohnkosten warm in €/Monat': 'rent_warm',
    'Leerstand in %': 'empty',
    'Kaufkraft je Haushalt in €/Monat': 'net_income',
    'Kaltmiete in % der Haushaltskaufkraft': 'rent_cold_fraction',
    'Wohnkostenquote in %': 'rent_warm_fraction',
    'Jahr': 'year',
    'Kaltmiete unteres Marktsegment in €/m2/Monat': 'sqm_price_lower',
    'Kaltmiete oberes Marktsegment in €/m2/Monat': 'sqm_price_upper',
    'Anzahl der Mietangebote': 'no_offers'
}

In [6]:
def rent_per_pr(df):
    df_rent = plz.merge(df, on='PLZ', how='outer')
    df_rent.rename(columns=col_mapping, inplace=True)
    df_rent = df_rent.applymap(lambda x: np.nan if x == 'k.A.' else x)
    return df_rent

In [7]:
rent_all = pd.DataFrame()

for key, value in rent.items():
    rent_all = rent_all.append(rent_per_pr(value), ignore_index=True)

In [8]:
#print(rent_all.loc[rent_all.apt_size.isna(), 'PR'].unique())
#print(rent_all.loc[rent_all.apt_size.isna(), 'PLZ'].unique())

In [9]:
rent_all.drop(['empty', 'no_offers'], axis=1, inplace=True)

In [10]:
rent_all['year'].fillna(method='backfill', inplace=True)

In [11]:
import joblib

joblib.dump(rent_all, './final_rent.csv')

['./final_rent.csv']